# Calcular los rezagos para las variables meteorológicas y para las variables epidemiológicas 


El script cargará el archivo desde la ubicación indicada, recorrerá **todas** las columnas numéricas (tanto `casos_dengue` como las variables meteorológicas de temperatura, humedad, precipitación, etc.) creando dinámicamente los rezagos (*lags*) desde la semana 1 hasta la 12, y finalmente aplicará una limpieza estricta eliminando las primeras 12 filas que lógicamente se quedan con valores vacíos (`NaN`).



Al terminar, guardará el resultado automáticamente en un archivo nuevo llamado `1_df_meteo_epi_lags12_limpio.xlsx` dentro de la misma carpeta.


In [1]:
import pandas as pd
import numpy as np

# 1. Definición de rutas de archivos
ruta_entrada = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\2_datos\1_raw\1_meteo_epi.xlsx"
ruta_salida = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\2_datos\1_raw\2_meteo_epi_rezagos_meteo_epi.xlsx"

print("Cargando el archivo de datos original...")
df = pd.read_excel(ruta_entrada)

# 2. Preprocesamiento temporal y ordenamiento cronológico
df['fecha'] = pd.to_datetime(df['fecha'])
df.set_index('fecha', inplace=True)
df = df.sort_index()

# 3. Identificar qué variables van a recibir la creación de rezagos
# Excluimos variables que solo sirven como identificadores de tiempo del registro actual
columnas_excluir = ['año', 'semana_epi']
columnas_para_lags = [col for col in df.columns if col not in columnas_excluir]

print(f"Generando rezagos (1 a 12 semanas) para {len(columnas_para_lags)} variables...")

# Diccionario auxiliar para acumular las nuevas columnas eficientemente
nuevas_columnas = {}

# 4. Bucle para calcular los rezagos de todas las variables seleccionadas
for col in columnas_para_lags:
    for i in range(1, 13):
        nombre_lag = f'{col}_lag_{i}'
        nuevas_columnas[nombre_lag] = df[col].shift(i)

# Convertimos el diccionario en Dataframe y lo concatenamos con el original
df_lags = pd.DataFrame(nuevas_columnas, index=df.index)
df = pd.concat([df, df_lags], axis=1)

# 5. Eliminación de instancias con valores nulos (NaN)
# Dado que el rezago máximo es de 12 semanas, las primeras 12 semanas de la serie temporal (inicio de 2021) 
# no tienen historial previo en el dataset y quedan vacías. Las eliminamos para limpiar el set.
print("Eliminando filas con registros nulos generados por el desplazamiento temporal...")
filas_antes = df.shape[0]
df.dropna(inplace=True)
filas_despues = df.shape[0]

print(f"-> Se eliminaron las primeras {filas_antes - filas_despues} filas debido a valores nulos.")

# 6. Guardar el archivo Excel resultante
print(f"Guardando el nuevo dataset expandido y limpio en: {ruta_salida}")
# Reseteamos el índice para que la columna 'fecha' vuelva a escribirse como columna normal en el Excel
df.reset_index().to_excel(ruta_salida, index=False)

print("\n=== PROCESO COMPLETADO EXITOSAMENTE ===")
print(f"Dimensiones finales del dataset: {df.shape[0]} filas (semanas) y {df.shape[1]} columnas (características).")


Cargando el archivo de datos original...
Generando rezagos (1 a 12 semanas) para 12 variables...
Eliminando filas con registros nulos generados por el desplazamiento temporal...
-> Se eliminaron las primeras 12 filas debido a valores nulos.
Guardando el nuevo dataset expandido y limpio en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\2_datos\1_raw\2_meteo_epi_rezagos_meteo_epi.xlsx

=== PROCESO COMPLETADO EXITOSAMENTE ===
Dimensiones finales del dataset: 249 filas (semanas) y 158 columnas (características).



```

### ¿Qué hace este script estructuralmente?

1. **Concatenación eficiente:** En lugar de agregar columna por columna al dataframe original dentro de un bucle pesado (lo cual genera advertencias de rendimiento en pandas), el script crea un set de datos temporal en memoria (`nuevas_columnas`) y luego lo une todo de un solo golpe con `pd.concat()`.
2. **`df.dropna(inplace=True)`**: Realiza la limpieza exacta que solicitaste. Purga de forma segura las semanas del inicio de la investigación (comienzo de 2021) que no contaban con 12 semanas previas de historia meteorológica para calcular sus respectivos retrasos. El resto del dataset queda completamente lleno e ideal para los algoritmos basados en árboles como LightGBM o XGBoost.